# Data Poisoning Attack
This notebook will walk you through performing a data poisoning attack on a computer vision model. We will be using the ResNet50 model with a traffic detection dataset to showcase the effect an attack like this can have on real-life applications of computer vision like autonomous driving. To conduct the attack, we will be using an imported library, the Adversarial Robustness Toolbox. This library contains a multitude of attacks and defenses for ML models.

# Setup


We have provided you with the model and dataset. The 'Import Dependencies' and 'Format Data' of this notebook have been completed for you. You may begin the exercise at the 'Load the Model' section after familiarizing yourself with the Setup.

## Import Dependencies

Installing the ART library.

In [ ]:
!pip install adversarial-robustness-toolbox

In [ ]:
import os, sys
from os.path import abspath

modulePath = os.path.abspath(os.path.join('..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)

import warnings
warnings.filterwarnings('ignore')


import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torchvision.io import decode_image
import torch.optim as optim

from art import config
from art.utils import load_dataset, get_file
from art.estimators.classification import PyTorchClassifier
from art.attacks.poisoning import FeatureCollisionAttack

import numpy as np

import pandas as pd

from PIL import Image

from pathlib import Path

import cv2

import csv

%matplotlib inline
import matplotlib.pyplot as plt

np.random.seed(301)

Grant access to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check the GPU available
print(f"Running on GPU: {torch.cuda.is_available()}")

## Format Data

Pull in dataset

In [ ]:
# TODO: Drag the this dataset file in the Data folder out to your My Drive directory.
!unzip './drive/My Drive/TrafficSignDataset.zip' &> /dev/null

Configure labeling

In [ ]:
classList = [
    "Speed limit (5km/h)",
    "Speed limit (15km/h)",
    "Speed limit (30km/h)",
    "Speed limit (40km/h)",
    "Speed limit (50km/h)",
    "Speed limit (60km/h)",
    "Speed limit (70km/h)",
    "Speed limit (80km/h)",
    "Dont Go straight or left",
    "Dont Go straight or Right",
    "Dont Go straight",
    "Dont Go Left",
    "Dont Go Left or Right",
    "Dont Go Right",
    "Dont overtake from Left",
    "No Uturn",
    "No Car",
    "No horn",
    "Go straight or right",
    "Go straight",
    "Go Left",
    "Go Left or right",
    "Go Right",
    "keep Left",
    "keep Right",
    "Roundabout mandatory",
    "watch out for cars",
    "Horn",
    "Bicycles crossing",
    "Uturn",
    "Road Divider",
    "Traffic signals",
    "Danger Ahead",
    "Zebra Crossing",
    "Bicycles crossing",
    "Children crossing",
    "Dangerous curve to the left",
    "Dangerous curve to the right",
    "Unknown1",
    "Unknown2",
    "Unknown3",
    "Go right or straight",
    "Go left or straight",
    "Unknown4",
    "ZigZag Curve",
    "Train Crossing",
    "Under Construction",
    "Unknown5",
    "Fences",
    "Heavy Vehicle Accidents",
    "Unknown6",
    "Give Way",
    "No stopping",
    "No entry",
    "Unknown7",
    "Unknown8",
]

Function for returning rows in CSV

In [ ]:
def findMatchingRow(csvFile, searchString, columnIndex=0):
    with open(csvFile, 'r') as file:
        reader = csv.reader(file)
        for row in reader:
            if row[columnIndex] == searchString:
                return row
    return None


Function for formatting data

In [ ]:
def processImages(dataDir, imgSize=(128, 128), isTraining = True):
    images = []
    labels = []
    for imgName in os.listdir(dataDir):
      # Import and normalize images
        imgPath = os.path.join(dataDir, imgName)
        img = cv2.imread(imgPath)
        img = cv2.resize(img, imgSize)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = img.transpose(2, 0, 1)

        images.append(img)

      # Add label to list
        if isTraining:
          row = findMatchingRow("./TrainingLabels.csv", imgName)
        else:
          row = findMatchingRow("./TestingLabels.csv", imgName)
        labels.append(int(row[1]))

    return np.array(images), np.array(labels)


Get test and training images and labels

In [ ]:
trainPath = "./TrainingFolder"
testPath = "./TestingFolder"
trainImages, trainLabels = processImages(trainPath)
testImages, testLabels = processImages(testPath, isTraining=False)

# Convert values to be between 0 and 1
trainImages = trainImages.astype('float32') / 255.0
testImages = testImages.astype('float32') / 255.0

In [ ]:
# Shape should be (# of images, # of color channels, dimentions of image)
print(trainImages.shape)

## Load the Model

Import pretrained model. We have already trained the model on your behalf to shorten the time completion of this exercise.

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

Trim last layer (feature layer) and apply our layer with the amount of classes needed.

In [ ]:
numClasses = 58
model.fc = nn.Linear(model.fc.in_features, numClasses)

Pull in model checkpoint

In [ ]:
# TODO: Put path to the project in your drive here
myFile = Path("./drive/My Drive/Data_Poisoning/bestModel.pth")
if myFile.is_file():
  model.load_state_dict(torch.load(myFile))

Convert model into PyTorch classifier. The library we are going to use for the attack, ART, uses this classifier.


In [ ]:
model.eval()

lossfnct =  nn.CrossEntropyLoss()
optalg = optim.Adam(model.parameters(), lr=0.0001)

#TODO: Complete the classifier function
classifier = PyTorchClassifier(clip_values=(0.0, 1.0),
                             model=model,
                             nb_classes=58,
                             input_shape=(3,128,128),
                            #Set loss function parameter (don't forget that comma)
                             loss= ,
                            #Set the optimization algorithm
                             optimizer=
                             )

Fit the classifier.

In [ ]:
classifier.fit(
    trainImages,
    trainLabels,
    nb_epochs=10, # You can mess with this later on your own time
    training_mode=True,
    batch_size=25,
    verbose=True
    )

Find example of victim class.

# Attack

Now that we have a clean trained model, time to configure and test our attack on it.

##Setup

For the attack we will need to pick our target class and then, our desired class.

In [ ]:
targetClass = 0 # Class to poison

firstIndex = np.where(testLabels == targetClass)[0][1]

targetInstance = np.expand_dims(testImages[firstIndex], axis=0)
imgPlot = np.transpose(targetInstance[0],(1,2,0))
fig = plt.imshow(imgPlot)
print("shape of targetInstance", targetInstance.shape)
print('trueClass: ' + classList[targetClass])
print('predictedClass: ' + classList[np.argmax(classifier.predict(targetInstance), axis=1)[0]])

featureLayer = classifier.layer_names[-2]
print(featureLayer)

Find all instances of the desired class

In [ ]:
baseClass = 7 # What we create the poison from
firstIndicies = np.where(trainLabels == baseClass)[0][:]
baseInstances = np.copy(trainImages[firstIndicies])
baseLabels = trainLabels[firstIndicies]
testPredictions = np.argmax(classifier.predict(baseInstances), axis = 1)
numbCorrectPred = np.sum(testPredictions == baseClass)

print("New test data to be poisoned (10 images):")
print("Correctly classified: {}".format(numbCorrectPred))
print("Incorrectly classified: {}".format(152-numbCorrectPred))

Show examples of desired class

In [ ]:
plt.figure(figsize=(10,10))
for i in range(0, 9):
  predLabel, trueLabel = classList[testPredictions[i]], classList[baseClass]
  plt.subplot(330 + 1 + i)
  fig=plt.imshow(np.transpose(baseInstances[i],(1,2,0)))
  fig.axes.get_xaxis().set_visible(False)
  fig.axes.get_yaxis().set_visible(False)
  fig.axes.text(0.5, -0.1, predLabel +  " (" + trueLabel + ")" , fontsize=8, transform=fig.axes.transAxes,
                  horizontalalignment='center')

##Configure Attack

Now that we have a working and accurate model, we need to poison it. This means modifying the data the model trains on by adding malicious labels and images for it, therefore, reducing its accuracy on clean data. If you want to know more, remember to look at the assignment document.


In [ ]:
# Configure attack with the ART library
attack = FeatureCollisionAttack(classifier,
                                targetInstance,
                                featureLayer,
                                max_iter=100,
                                similarity_coeff=256,
                                watermark=0.3, # The potency of the perturbation on the clean images.
                                learning_rate=1)

# Create poison instances: Remember this!
poison, poisonLabels = attack.poison(baseInstances)

Display 10 poison images

In [ ]:
poisonPred = np.argmax(classifier.predict(poison), axis=1)
plt.figure(figsize=(10,10))
for i in range(0, 9):
    predLabel, trueLabel = classList[poisonPred[i]], classList[np.argmax(poisonLabels[i])]
    plt.subplot(330 + 1 + i)
    fig=plt.imshow(np.transpose(poison[i],(1,2,0)))
    fig.axes.get_xaxis().set_visible(False)
    fig.axes.get_yaxis().set_visible(False)
    fig.axes.text(0.5, -0.1, predLabel + " (" + trueLabel + ")", fontsize=8, transform=fig.axes.transAxes,
                  horizontalalignment='center')

## Train with poison

Poison labels are a list of confidences for each class on each label. The model trained using single number label, so we need to convert to a list of the poison class number

In [ ]:
poisonLabelsList = []
for i in poisonLabels:
  poisonLabelsList.append(np.argmax(i))
poisonLabels = np.array(poisonLabelsList)

We will be training the model on just the poisoned data for this exercise to showcase the effects of data poisoning. Like above we used clean images and labels now we need to train with the poisoned versions.

In [ ]:
model.train()

classifier.fit(
    #TODO: Pass in the poison instances of images and labels as the function parameters.
     ,
     ,
    nb_epochs=20,
    batch_size=2,
    verbose=True
)

# Test the attack

If successful, the model should now be outputting incorrect labels for images pertaining to the poisoned class.

In [ ]:
fig= plt.imshow(np.transpose(targetInstance[0],(1,2,0)))
fig.axes.get_xaxis().set_visible(False)
prediction = classifier.predict(targetInstance)

print('true class: ' + classList[targetClass])
print('predicted class: ' + classList[np.argmax(prediction)])